<a href="https://colab.research.google.com/github/TRUPALIX9/card-snap-backend/blob/main/train_florence2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup Florence-2 Fine-Tuning Environment
Run this cell to install all required High-Performance GPU libraries.

In [ ]:
!pip install -q datasets flash_attn timm einops
!pip install -q transformers[torch] peft
!pip install -q bitsandbytes accelerate

  Preparing metadata (setup.py) ... done


# 2. Mount Google Drive
If you uploaded your `dataset` folder to Google Drive, run this line so Colab can read your images and Annotations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change directory to where your dataset is (Update this path!)
# %cd /content/drive/MyDrive/card-reader-ai_scripts

# 3. Start Training Loop!
Run this block to initialize LoRA on Florence-2 and train it against your `annotations.json`!

In [ ]:
import os
import json
import torch
from datasets import Dataset
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# ==========================================
# 1. Configuration
# ==========================================
MODEL_ID = "microsoft/Florence-2-base"
DATASET_DIR = "./dataset"
ANNOTATIONS_FILE = "./dataset/annotations.json"
OUTPUT_DIR = "./custom_florence2_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\ud83d\ude80 Initializing Florence-2 Fine-Tuning Pipeline on {device.upper()}...")

# ==========================================
# 2. Prepare the Dataset
# ==========================================
def load_dataset():
    print("\ud83d\udce5 Loading images and annotations...")
    with open(ANNOTATIONS_FILE, "r") as f:
        annotations = json.load(f)

    data_list = []
    for filename, json_string in annotations.items():
        image_path = os.path.join(DATASET_DIR, filename)
        if os.path.exists(image_path):
            data_list.append({
                "image_path": image_path,
                "text": json_string
            })

    print(f"\u2705 Loaded {len(data_list)} highly accurate training examples.")
    return Dataset.from_list(data_list)

dataset = load_dataset()

# ==========================================
# 3. Load Model and Processor
# ==========================================
print(f"\ud83e\udde0 Loading {MODEL_ID} Processor and Model...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)

# ==========================================
# 4. LoRA Setup (Low-Rank Adaptation)
# ==========================================
print("\u2699\ufe0f Applying LoRA Configuration to base model...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "linear", "Conv2d"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==========================================
# 5. Data Formatting (Collate Function)
# ==========================================
def collate_fn(batch):
    texts = [f"<OCR_JSON> Extract structured contact information from this visiting card.\n{item['text']}" for item in batch]
    images = [Image.open(item['image_path']).convert("RGB") for item in batch]

    max_size = 768
    processed_images = []
    for img in images:
        if max(img.size) > max_size:
            img.thumbnail((max_size, max_size))
        processed_images.append(img)

    inputs = processor(
        text=texts,
        images=processed_images,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

# ==========================================
# 6. Training Configuration
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=15,
    save_steps=100,
    logging_steps=10,
    fp16=(device == "cuda"),
    remove_unused_columns=False,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn
)

# ==========================================
# 7. Start Training
# ==========================================
print("\n" + "="*50)
print("\ud83d\udd25 STARTING FINE-TUNING \ud83d\udd25")
print("="*50 + "\n")

trainer.train()

print("\n\u2705 Training Complete!")
print(f"\ud83d\udcbe Saving your custom model to: {OUTPUT_DIR}")

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("\ud83c\udf89 Done! Zip the 'custom_florence2_model' folder and download it back to your PC!")
